# Compose main modelling table (synthetic data)

Synthetic-data equivalent of `paper-compose_main_modelling_table-proper_loso-expanded-session_level.ipynb`.

Takes the session-level analysis results produced by `evaluate_session_level_ccc.py`
and formats them as a LaTeX table suitable for the publication.

**Input:** `results/synthetic/composed/session_level_analysis/compiled-session_level-paper_configs.csv`  
**Output:** LaTeX source printed in the final cell.

In [1]:
import os
import pandas as pd

path_results_base = "../../results/synthetic/composed/session_level_analysis"
table_paper = "compiled-session_level-paper_configs.csv"

table_paper_path = os.path.join(path_results_base, table_paper)

df_paper = pd.read_csv(table_paper_path)
df_paper_raw = df_paper.copy()
print(f"Loaded {len(df_paper)} rows.")
df_paper.head()

Loaded 10 rows.


,ccc_conf_mean,ccc_conf_low,ccc_conf_high,lower_bound_larger_null,task,Features,ccc_segment_mean,ccc_segment_low,ccc_segment_high,concordance_cc-test-agg-average,path,Target,Task,Survey
0,0.018016,-0.017422,0.048588,False,speechtasks-standardized_tasks,eGeMAPSv02,0.017421,-0.021716,0.054670,0.018016,/who_5_percentage_score_corrected/raw_0_100-no...,who_5_percentage_score_corrected,speechtasks-standardized_tasks,synthetic-noisy
1,0.034147,-0.004983,0.074949,False,speechtasks-standardized_tasks,wav2vec2-variant-wav2vec2-large-robust-12-ft-e...,0.032685,-0.011144,0.076176,0.034147,/who_5_percentage_score_corrected/raw_0_100-no...,who_5_percentage_score_corrected,speechtasks-standardized_tasks,synthetic-noisy
2,-0.008704,-0.043531,0.026874,False,speechtasks-standardized_tasks,eGeMAPSv02,-0.008479,-0.050181,0.032561,-0.008704,/pss_10_total_score/raw_0_40-normalized_0_1/sp...,pss_10_total_score,speechtasks-standardized_tasks,synthetic-noisy
3,-0.002736,-0.038794,0.031079,False,speechtasks-standardized_tasks,wav2vec2-variant-wav2vec2-large-robust-12-ft-e...,-0.002647,-0.038780,0.037450,-0.002736,/pss_10_total_score/raw_0_40-normalized_0_1/sp...,pss_10_total_score,speechtasks-standardized_tasks,synthetic-noisy
4,-0.014765,-0.047731,0.011647,False,speechtasks-standardized_tasks,eGeMAPSv02,-0.014538,-0.042767,0.011273,-0.014765,/phq_8_total_score/raw_0_24-normalized_0_1/spe...,phq_8_total_score,speechtasks-standardized_tasks,synthetic-noisy


In [2]:
# Replace strings in the "Target" column
df_paper["Target"] = df_paper["Target"].apply(
    lambda x: (
        "WHO-5"
        if "who_5_percentage_score_corrected" in x
        else (
            "PSS-10"
            if "pss_10_total_score" in x
            else (
                "Stress-now"
                if "stress_current" in x
                else (
                    "PHQ-8"
                    if "phq_8_total_score" in x
                    else ("Stress-work" if "stress_work_tasks" in x else x)
                )
            )
        )
    )
)

In [3]:
# Replace strings in the "Task" column
df_paper["Task"] = df_paper["Task"].apply(
    lambda x: ("All" if "speechtasks-standardized_tasks" in str(x) else str(x))
)

In [4]:
# Replace strings in the "Features" column
df_paper["Features"] = df_paper["Features"].apply(
    lambda x: (
        "eGeMAPS"
        if "eGeMAPSv02" in str(x)
        else ("W2V2" if "wav2vec2" in str(x) else str(x))
    )
)

In [5]:
# Extract the "Model" from the "path" column
df_paper["Model"] = df_paper["path"].str.extract(
    r"/([^/]+)/models/results-compiled\.yaml$"
)
df_paper["Model"] = df_paper["Model"].apply(
    lambda x: (
        "RF"
        if "RandomForestRegressor" in str(x)
        else (
            "XGBr"
            if "XGBRegressor" in str(x)
            else (
                "LR"
                if "LinearRegression" in str(x)
                else ("SVR" if "SVR" in str(x) else str(x))
            )
        )
    )
)

In [6]:
# Summarize CCC with confidence intervals into one column
df_paper["CCC"] = df_paper.apply(
    lambda row: f"${row['ccc_conf_mean']:.3f} ({row['ccc_conf_low']:.3f} - {row['ccc_conf_high']:.3f})$",
    axis=1,
)

# Mark significant results
df_paper.loc[df_paper["lower_bound_larger_null"] == True, "CCC"] += "*"

In [7]:
# Define sort order
target_order = ["WHO-5", "PSS-10", "PHQ-8", "Stress-now", "Stress-work"]
df_paper["Target"] = pd.Categorical(
    df_paper["Target"], categories=target_order, ordered=True
)
df_paper = df_paper.sort_values(by=["Target", "Features"])

# Keep only the columns relevant for LaTeX output
df_paper_table = df_paper[["Target", "CCC", "Task", "Model", "Features"]]
df_paper_table

,Target,CCC,Task,Model,Features
1,WHO-5,$0.034 (-0.005 - 0.075)$,All,RF,W2V2
0,WHO-5,$0.018 (-0.017 - 0.049)$,All,RF,eGeMAPS
3,PSS-10,$-0.003 (-0.039 - 0.031)$,All,RF,W2V2
2,PSS-10,$-0.009 (-0.044 - 0.027)$,All,RF,eGeMAPS
5,PHQ-8,$-0.012 (-0.052 - 0.030)$,All,RF,W2V2
4,PHQ-8,$-0.015 (-0.048 - 0.012)$,All,RF,eGeMAPS
7,Stress-now,$0.000 (-0.037 - 0.032)$,All,RF,W2V2
6,Stress-now,$-0.022 (-0.046 - 0.001)$,All,RF,eGeMAPS
9,Stress-work,$-0.032 (-0.062 - 0.002)$,All,RF,W2V2
8,Stress-work,$-0.000 (-0.036 - 0.039)$,All,RF,eGeMAPS


In [8]:
latex_code = df_paper_table.to_latex(index=False)
print(latex_code)

\begin{tabular}{lllll}
\toprule
Target & CCC & Task & Model & Features \\
\midrule
WHO-5 & $0.034 (-0.005 - 0.075)$ & All & RF & W2V2 \\
WHO-5 & $0.018 (-0.017 - 0.049)$ & All & RF & eGeMAPS \\
PSS-10 & $-0.003 (-0.039 - 0.031)$ & All & RF & W2V2 \\
PSS-10 & $-0.009 (-0.044 - 0.027)$ & All & RF & eGeMAPS \\
PHQ-8 & $-0.012 (-0.052 - 0.030)$ & All & RF & W2V2 \\
PHQ-8 & $-0.015 (-0.048 - 0.012)$ & All & RF & eGeMAPS \\
Stress-now & $0.000 (-0.037 - 0.032)$ & All & RF & W2V2 \\
Stress-now & $-0.022 (-0.046 - 0.001)$ & All & RF & eGeMAPS \\
Stress-work & $-0.032 (-0.062 - 0.002)$ & All & RF & W2V2 \\
Stress-work & $-0.000 (-0.036 - 0.039)$ & All & RF & eGeMAPS \\
\bottomrule
\end{tabular}

